# Retrieval-Augmented Generation (RAG) with LangChain & Gemini

**RAG** is a technique that improves LLM responses by first retrieving relevant information from a knowledge base, then using that context to generate a grounded answer.

The pipeline has three stages:
1. **Embed** — convert your documents into vector representations
2. **Retrieve** — find the most relevant documents for a given query
3. **Generate** — pass the retrieved context to an LLM to produce a final answer

```
User Query ──► Embeddings ──► Vector Search ──► Top-K Docs ──► LLM ──► Answer
```

In [ ]:
# Install required packages
!pip install langchain langchain-google-genai langchain-community faiss-cpu python-dotenv

## Imports

- `langchain_google_genai` — Gemini embeddings + chat model via LangChain
- `langchain_community.vectorstores.FAISS` — in-memory vector store for similarity search
- `langchain_core` — prompt templates, output parsers, and the LCEL (LangChain Expression Language) `RunnablePassthrough`

In [ ]:
import os
from dotenv import load_dotenv

from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

# langchain_google_genai reads GOOGLE_API_KEY from the environment
os.environ["GOOGLE_API_KEY"] = os.getenv("gemini_API", "")

## Knowledge Base

This is the private data we want the model to reason over — information it was never trained on. In a real app this could come from PDFs, databases, or web pages.

We wrap each piece of text in a `Document` object so LangChain can work with it uniformly.

In [ ]:
raw_texts = [
    # ===== Annual Trends =====
    "Netflix generated approximately $33.7 billion in revenue in 2023, driven by steady global subscriber growth and strong engagement with original content.",
    "In 2024, Netflix revenue increased to approximately $39 billion, supported by password-sharing enforcement and expansion of ad-supported subscription plans.",
    "Early industry projections estimate Netflix revenue in 2025 may exceed $42 billion if ad-supported plans and international markets continue to grow.",
    # ===== 2023 Quarterly Trends =====
    "Netflix Q1 2023 revenue was approximately $8.16 billion, reflecting stable subscriber growth at the beginning of the year.",
    "Netflix Q2 2023 revenue was approximately $8.19 billion as international content helped sustain viewer engagement.",
    "Netflix Q3 2023 revenue increased to approximately $8.54 billion following successful original series releases.",
    "Netflix Q4 2023 revenue reached approximately $8.83 billion due to increased holiday viewing and seasonal content launches.",
    # ===== 2024 Quarterly Trends =====
    "Netflix Q1 2024 revenue increased to approximately $9.38 billion, driven by adoption of ad-supported subscription tiers.",
    "Netflix Q2 2024 revenue reached approximately $9.56 billion as password-sharing restrictions converted free users into paying subscribers.",
    "Netflix Q3 2024 revenue grew to approximately $9.82 billion, supported by strong engagement with global original content.",
    "Netflix Q4 2024 revenue peaked at approximately $10.25 billion, driven by franchise releases and high subscriber retention.",
    # ===== 2025 Early Signals & Projections =====
    "Early 2025 trends indicate Netflix revenue growth is shifting from subscriber expansion to increased monetization per user.",
    "Ad-supported subscription tiers are expected to become a major revenue driver for Netflix in 2025.",
    "Netflix's 2025 growth strategy emphasizes content localization to expand viewership in emerging markets.",
    "Market analysts expect Netflix revenue growth in 2025 to be moderate rather than explosive due to saturation in mature streaming markets.",
    # ===== Behavioral & Strategic Insights =====
    "Netflix revenue patterns show strong seasonality, with higher performance in Q3 and Q4 driven by content release timing.",
    "Consumer willingness to accept ads in exchange for lower subscription prices has improved Netflix's revenue resilience.",
    "Retention and engagement metrics are becoming more important than raw subscriber growth for Netflix's long-term sustainability.",
]

# Wrap each string in a Document
documents = [Document(page_content=text) for text in raw_texts]
print(f"Loaded {len(documents)} documents into the knowledge base.")

## Step 1 — Build the Vector Store

`GoogleGenerativeAIEmbeddings` converts each document into a dense vector (a list of numbers that captures semantic meaning). FAISS stores those vectors and lets us do fast similarity search.

`FAISS.from_documents()` does both steps in one call: it embeds every document and indexes them.

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

vectorstore = FAISS.from_documents(documents, embeddings)

# A retriever wraps the vector store and handles the similarity search for us
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Vector store ready. Retriever will return top 3 matches per query.")

## Step 2 — Build the RAG Chain

LangChain Expression Language (LCEL) lets us pipe components together with `|`, much like Unix pipes.

Our chain:
```
query ──► {context: retriever, question: passthrough} ──► prompt ──► llm ──► string
```

- `retriever | format_docs` fetches the top-3 documents and joins them into a single string
- `RunnablePassthrough()` passes the original query through unchanged
- The prompt injects both into a message for Gemini
- `StrOutputParser()` extracts the plain text from the model's response

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

prompt = ChatPromptTemplate.from_template("""
You are a digital media consumption researcher analyzing streaming platforms.
Use ONLY the context below to answer the question.
If the answer is not in the context, say so clearly.

Context:
{context}

Question:
{question}

Answer:
""")

def format_docs(docs):
    """Join retrieved Document objects into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain is ready.")

## Step 3 — Ask Questions

Now we can query the pipeline. Each call to `rag_chain.invoke(query)` will:
1. Embed the query
2. Retrieve the 3 most relevant documents
3. Pass them + the query to Gemini
4. Return Gemini's grounded answer

Type `q` to quit.

In [ ]:
while True:
    query = input("\n---\nEnter your question about Netflix (q to quit): ")
    if query.strip().lower() == "q":
        break

    # Retrieve context separately so we can display it
    retrieved_docs = retriever.invoke(query)
    context_preview = "\n".join(f"  - {doc.page_content}" for doc in retrieved_docs)
    print(f"\nRetrieved context:\n{context_preview}")

    answer = rag_chain.invoke(query)
    print(f"\nAnswer:\n{answer}")